In [14]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%autosave 0
    
import logging
logging.basicConfig(format='%(asctime)s %(levelname)s:%(message)s', level=logging.DEBUG, datefmt='%I:%M:%S')

import os
import sys
import numpy as np
import pandas as pd
import sdata
import uuid
import time
from sdata import Data
from sdata.iolib.vault import FileSystemVault
from sdata.iolib.hdf import FlatHDFDataStore
print("sdata v{}".format(sdata.__version__))
from sdata.contrib.simple_graph import database as db

09:01:37 DEBUG:Loaded backend module://ipykernel.pylab.backend_inline version unknown.


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Autosave disabled
sdata v0.17.0


In [18]:
db_name = "index.sqlite"

In [19]:
os.remove(db_name)

In [20]:

db.initialize(db_name)

In [21]:
db.atomic(db_name, db.add_node({'name': 'apple Computer Company', 'type':['company', 'start-up'], 'founded': 'April 1, 1976'}, 1))
db.atomic(db_name, db.add_node({'name': 'Steve Wozniak', 'type':['person','engineer','founder']}, 2))
db.atomic(db_name, db.add_node({'name': 'Steve Jobs', 'type':['person','designer','founder']}, 3))
db.atomic(db_name, db.add_node({'name': 'Ronald Wayne', 'type':['person','administrator','founder']}, 4))
db.atomic(db_name, db.add_node({'name': 'Mike Markkula', 'type':['person','investor']}, 5))
db.atomic(db_name, db.connect_nodes(2, 1, {'action': 'founded'}))
db.atomic(db_name, db.connect_nodes(3, 1, {'action': 'founded'}))
db.atomic(db_name, db.connect_nodes(4, 1, {'action': 'founded'}))
db.atomic(db_name, db.connect_nodes(5, 1, {'action': 'invested', 'equity': 80000, 'debt': 170000}))
db.atomic(db_name, db.connect_nodes(1, 4, {'action': 'divested', 'amount': 800, 'date': 'April 12, 1976'}))
db.atomic(db_name, db.connect_nodes(2, 3))
db.atomic(db_name, db.upsert_node(2, {'nickname': 'Woz'}))


!!!results [('{"name":"Steve Wozniak","type":["person","engineer","founder"],"id":2}',)]


In [22]:
db.atomic(db_name, db.find_node(1))

!!!results [('{"name":"apple Computer Company","type":["company","start-up"],"founded":"April 1, 1976","id":1}',)]


{'name': 'apple Computer Company',
 'type': ['company', 'start-up'],
 'founded': 'April 1, 1976',
 'id': 1}

In [23]:
db.atomic(db_name, db.find_nodes({'name': 'Steve'}, db._search_like, db._search_starts_with))

!!!results [('{"name":"Steve Wozniak","type":["person","engineer","founder"],"id":2,"nickname":"Woz"}',), ('{"name":"Steve Jobs","type":["person","designer","founder"],"id":3}',)]


[{'name': 'Steve Wozniak',
  'type': ['person', 'engineer', 'founder'],
  'id': 2,
  'nickname': 'Woz'},
 {'name': 'Steve Jobs', 'type': ['person', 'designer', 'founder'], 'id': 3}]

In [24]:
db.traverse(db_name, 2, 3)
db.traverse(db_name, 4, 5)

!!!results [('{"name":"Steve Wozniak","type":["person","engineer","founder"],"id":2,"nickname":"Woz"}',)]
!!!results [('2', '1', '{"action":"founded"}'), ('2', '3', '{}')]
!!!results [('2', '1', '{"action":"founded"}'), ('2', '3', '{}')]
!!!results [('{"name":"apple Computer Company","type":["company","start-up"],"founded":"April 1, 1976","id":1}',)]
!!!results [('{"name":"Steve Wozniak","type":["person","engineer","founder"],"id":2,"nickname":"Woz"}',)]
!!!results [('{"name":"Steve Jobs","type":["person","designer","founder"],"id":3}',)]
!!!results [('{"name":"Ronald Wayne","type":["person","administrator","founder"],"id":4}',)]
!!!results [('4', '1', '{"action":"founded"}'), ('1', '4', '{"action":"divested","amount":800,"date":"April 12, 1976"}')]
!!!results [('4', '1', '{"action":"founded"}'), ('1', '4', '{"action":"divested","amount":800,"date":"April 12, 1976"}')]
!!!results [('{"name":"apple Computer Company","type":["company","start-up"],"founded":"April 1, 1976","id":1}',)]
!!!

[4, 1, 5]

In [25]:
db.traverse(db_name, 5, neighbors_fn=db.find_inbound_neighbors)

!!!results [('{"name":"Mike Markkula","type":["person","investor"],"id":5}',)]
!!!results []
!!!results []


[5]

In [26]:
db.traverse(db_name, 5, neighbors_fn=db.find_outbound_neighbors)

!!!results [('{"name":"Mike Markkula","type":["person","investor"],"id":5}',)]
!!!results [('5', '1', '{"action":"invested","equity":80000,"debt":170000}')]
!!!results [('5', '1', '{"action":"invested","equity":80000,"debt":170000}')]
!!!results [('{"name":"apple Computer Company","type":["company","start-up"],"founded":"April 1, 1976","id":1}',)]
!!!results [('{"name":"Mike Markkula","type":["person","investor"],"id":5}',)]
!!!results [('1', '4', '{"action":"divested","amount":800,"date":"April 12, 1976"}')]
!!!results [('1', '4', '{"action":"divested","amount":800,"date":"April 12, 1976"}')]
!!!results [('{"name":"apple Computer Company","type":["company","start-up"],"founded":"April 1, 1976","id":1}',)]
!!!results [('{"name":"Ronald Wayne","type":["person","administrator","founder"],"id":4}',)]
!!!results [('4', '1', '{"action":"founded"}')]
!!!results [('4', '1', '{"action":"founded"}')]
!!!results [('{"name":"apple Computer Company","type":["company","start-up"],"founded":"April 1

[5, 1, 4]

In [27]:
db.traverse(db_name, 5, neighbors_fn=db.find_neighbors)

!!!results [('{"name":"Mike Markkula","type":["person","investor"],"id":5}',)]
!!!results [('5', '1', '{"action":"invested","equity":80000,"debt":170000}')]
!!!results [('5', '1', '{"action":"invested","equity":80000,"debt":170000}')]
!!!results [('{"name":"apple Computer Company","type":["company","start-up"],"founded":"April 1, 1976","id":1}',)]
!!!results [('{"name":"Mike Markkula","type":["person","investor"],"id":5}',)]
!!!results [('1', '4', '{"action":"divested","amount":800,"date":"April 12, 1976"}'), ('2', '1', '{"action":"founded"}'), ('3', '1', '{"action":"founded"}'), ('4', '1', '{"action":"founded"}'), ('5', '1', '{"action":"invested","equity":80000,"debt":170000}')]
!!!results [('1', '4', '{"action":"divested","amount":800,"date":"April 12, 1976"}'), ('2', '1', '{"action":"founded"}'), ('3', '1', '{"action":"founded"}'), ('4', '1', '{"action":"founded"}'), ('5', '1', '{"action":"invested","equity":80000,"debt":170000}')]
!!!results [('{"name":"apple Computer Company","typ

[5, 1, 4, 3, 2]

In [10]:
db.visualize(db_name, '{}.dot'.format(db_name), [4, 1, 5])

In [11]:
os.system("dot -Tpng {0}.dot -o {0}.png".format(db_name))

0

In [12]:
db.visualize(db_name, '{}_no_types.dot'.format(db_name), [4, 1, 5], exclude_node_keys=['type'], hide_edge_key=True)

In [13]:
os.system("dot -Tpng {0}_no_types.dot -o {0}_no_types.png".format(db_name))


0